# Notebook 10 — Adaptive vs Fixed Mutation Schedule
**Novelty:** Adaptive c3 schedule for MIGA (exploration → exploitation).

The base paper uses a **fixed** `c3` throughout all G generations: every
generation produces `c1 × c3` mutation offspring.  We propose a **linear
decay schedule** — starting high (broad exploration of gene space) and
decaying to a small value (fine exploitation near the elite):

$$c_3(g) = \text{round}\!\left(c_3^{\text{start}} + (c_3^{\text{end}} - c_3^{\text{start}}) \cdot \frac{g}{G-1}\right)$$

**Motivation:** Early generations benefit from many diverse mutation
offspring to escape local minima.  Late generations benefit from fewer
but higher-quality offspring produced from an already-refined elite pool.
This mirrors the exploration-exploitation tradeoff in evolutionary
computation (Holland 1975; Goldberg 1989).

**Datasets:** Iris, Glass, Haberman — chosen because they span different
n/p ratios and feature types.  Wholesale and Wine are left out: Wholesale
has high variance across seeds; Wine has a fundamental rank-deficiency
issue documented in Notebook 03.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from miga import MIGA
from miga.fitness import FitnessEvaluator
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance
from miga.data_utils import apply_mar, apply_mnar, auto_generators, compute_metrics, EXCLUDE_COLS
from miga.paper_results import (
    TABLE3_PARAMS, BENCHMARK_Q,
    TABLE4_RMSE, TABLE5_MAD, TABLE6_COD,
    METHODS, PERCENTAGES,
)

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

## 1. Configuration

In [ ]:
# Datasets to compare
from miga.data_utils import load_iris, load_glass, load_haberman

DATASETS = {
    "Iris":     (load_iris,     {"pct": 30, "seed": 30, "exclude_cols": set()}),
    "Glass":    (load_glass,    {"pct": 30, "seed": 30, "exclude_cols": {9}}),
    "Haberman": (load_haberman, {"pct": 30, "seed": 30, "exclude_cols": set()}),
}

# GA parameters (reduced for speed; scale up for final thesis runs)
L, G_GENS, Q_RUNS = 200, 500, 5

# Fixed baseline
C3_FIXED = 5

# Adaptive schedule: c3 decays from C3_START → C3_END over G generations
C3_START, C3_END = 15, 3

SEED = 42
print(f"l={L}, G={G_GENS}, Q={Q_RUNS}")
print(f"Fixed: c3={C3_FIXED}")
print(f"Adaptive: c3 {C3_START} → {C3_END}")

## 2. c3 Schedule Visualisation

In [ ]:
gens = np.arange(G_GENS)
c3_curve = np.round(C3_START + (C3_END - C3_START) * gens / max(G_GENS - 1, 1)).astype(int)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(gens, c3_curve, color="tab:blue", lw=2, label=f"Adaptive c3 ({C3_START}→{C3_END})")
ax.axhline(C3_FIXED, color="tab:orange", lw=2, ls="--", label=f"Fixed c3={C3_FIXED}")
ax.set_xlabel("Generation")
ax.set_ylabel("c3 (mutation offspring per parent)")
ax.set_title("Adaptive vs Fixed Mutation Rate Schedule")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "10_c3_schedule.png"), dpi=150)
plt.show()
print("Saved: results/10_c3_schedule.png")

## 3. Load Results

Run each dataset in a **separate terminal** (in parallel):
```
.venv/bin/python scripts/run_adaptive_dataset.py Iris
.venv/bin/python scripts/run_adaptive_dataset.py Glass
.venv/bin/python scripts/run_adaptive_dataset.py Haberman
```
Each script saves `results/10_adaptive_<dataset>.json`.
Re-run this cell after all three finish to load results.

In [ ]:
import json, os

DATASET_NAMES = ["Iris", "Glass", "Haberman"]
results = {}

for ds_name in DATASET_NAMES:
    path = os.path.join(RESULTS_DIR, f"10_adaptive_{ds_name.lower()}.json")
    if not os.path.exists(path):
        print(f"  [MISSING] {path} — run the script for {ds_name} first")
        continue
    with open(path) as f:
        data = json.load(f)
    results[ds_name] = data
    fixed_rmse    = data["fixed"]["rmse"]
    adaptive_rmse = data["adaptive"]["rmse"]
    delta = (fixed_rmse - adaptive_rmse) / fixed_rmse * 100
    print(f"{ds_name:10s}  fixed={fixed_rmse:.4f}  adaptive={adaptive_rmse:.4f}  Δ={delta:+.1f}%")

print(f"\nLoaded {len(results)}/{len(DATASET_NAMES)} datasets.")

## 4. Convergence Plots (Fr vs Generation)

In [ ]:
if not results:
    print("No results loaded yet. Run the scripts first.")
else:
    n_ds = len(results)
    fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
    if n_ds == 1:
        axes = [axes]

    for ax, (ds_name, data) in zip(axes, results.items()):
        for label, colour, ls in [("fixed", "tab:orange", "--"), ("adaptive", "tab:blue", "-")]:
            gen_hists = data[label]["gen_history"]   # list of Q lists
            arr = np.array(gen_hists)                 # (Q, G)
            mean_curve = arr.mean(axis=0)
            std_curve  = arr.std(axis=0)
            gens = np.arange(arr.shape[1])
            ax.plot(gens, mean_curve, color=colour, ls=ls, lw=2,
                    label=f"{label} (Q={len(gen_hists)})")
            ax.fill_between(gens,
                            mean_curve - std_curve,
                            mean_curve + std_curve,
                            alpha=0.15, color=colour)

        ax.set_title(ds_name)
        ax.set_xlabel("Generation")
        ax.set_ylabel("Best F_r")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.suptitle("Convergence: Adaptive vs Fixed c3 Schedule", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "10_convergence.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/10_convergence.png")

## 5. RMSE Comparison Table

In [ ]:
from miga.paper_results import TABLE4_RMSE

if results:
    rows = []
    for ds_name, data in results.items():
        paper_rmse    = TABLE4_RMSE.get(ds_name, {}).get(30, float("nan"))
        fixed_rmse    = data["fixed"]["rmse"]
        adaptive_rmse = data["adaptive"]["rmse"]
        delta         = (fixed_rmse - adaptive_rmse) / fixed_rmse * 100
        rows.append({
            "Dataset":     ds_name,
            "Paper RMSE":  f"{paper_rmse:.4f}",
            "Fixed c3":    f"{fixed_rmse:.4f}  ({fixed_rmse/paper_rmse:.2f}x)",
            "Adaptive c3": f"{adaptive_rmse:.4f}  ({adaptive_rmse/paper_rmse:.2f}x)",
            "Delta RMSE":  f"{'+'if delta>0 else ''}{delta:.1f}%",
        })
    df = pd.DataFrame(rows).set_index("Dataset")
    display(df)

## 6. Best F_r per Run

In [ ]:
if results:
    n_ds = len(results)
    fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4))
    if n_ds == 1:
        axes = [axes]

    for ax, (ds_name, data) in zip(axes, results.items()):
        fixed_scores    = data["fixed"]["history"]
        adaptive_scores = data["adaptive"]["history"]
        x = np.arange(len(fixed_scores))
        ax.bar(x - 0.2, fixed_scores,    0.35, label="Fixed",    color="tab:orange", alpha=0.8)
        ax.bar(x + 0.2, adaptive_scores, 0.35, label="Adaptive", color="tab:blue",   alpha=0.8)
        ax.set_title(ds_name)
        ax.set_xlabel("Run index")
        ax.set_ylabel("Best F_r")
        ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    plt.suptitle("Best F_r per Run: Fixed vs Adaptive c3", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "10_best_fr_per_run.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 7. Summary

In [ ]:
if results:
    print("=" * 60)
    print("SUMMARY — Adaptive vs Fixed c3 Schedule")
    print("=" * 60)
    for ds_name, data in results.items():
        f_rmse = data["fixed"]["rmse"]
        a_rmse = data["adaptive"]["rmse"]
        f_fr   = data["fixed"]["best_score"]
        a_fr   = data["adaptive"]["best_score"]
        delta_rmse = (f_rmse - a_rmse) / f_rmse * 100
        delta_fr   = (f_fr   - a_fr)   / f_fr   * 100
        winner = "Adaptive" if a_rmse < f_rmse else "Fixed"
        print(f"\n{ds_name}:")
        print(f"  RMSE   Fixed={f_rmse:.4f}  Adaptive={a_rmse:.4f}  D={delta_rmse:+.1f}%  [{winner}]")
        print(f"  Best Fr Fixed={f_fr:.4f}  Adaptive={a_fr:.4f}  D={delta_fr:+.1f}%")